# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema available at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema contains all information about available record sets, fields, and how to access tabular data files.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for the FAIR^2 dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (metadata only at this stage)
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print(f"Dataset loaded: {getattr(metadata, 'name', 'Unknown dataset')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Let's inspect the available record sets, fields, and their corresponding `@id` references, which we will need for further data extraction.

In [ ]:
# List all RecordSets in the metadata
record_sets = getattr(metadata, 'record_sets', None) or getattr(metadata, 'record_set', [])

print('Record sets available in the dataset:')
for rs in record_sets:
    recset_id = getattr(rs, '@id', None)
    recset_name = getattr(rs, 'name', None)
    print(f"  RecordSet: {recset_name}, @id: {recset_id}")

# For further exploration, print the fields for each record set
print('\nFields per RecordSet:')
record_sets_ids = []
for rs in record_sets:
    recset_id = getattr(rs, '@id', None)
    record_sets_ids.append(recset_id)
    recset_name = getattr(rs, 'name', None)
    print(f'--- {recset_name} (@id={recset_id}) ---')
    for field in getattr(rs, 'fields', []):
        fname = getattr(field, 'name', '')
        fid = getattr(field, '@id', '')
        dtype = getattr(field, 'data_type', '')
        print(f"  Field: {fname} (@id={fid}) type={dtype}")

## 3. Data Extraction
We'll load data from each record set into DataFrames. All record set and field references are handled by their `@id` values for programmatic reference.

In [ ]:
# Parse data from all available record sets
dataframes = {}

print(f"RecordSet IDs available: {record_sets_ids}\n")

for recset_id in record_sets_ids:
    # Use the `@id` for each RecordSet to extract records
    print(f"Loading records from RecordSet @id: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    if len(records) == 0:
        print(f"  No records found for {recset_id}\n")
        continue
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  Columns: {list(df.columns)}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Select a numeric field (by its `@id`, e.g. a "Molecular Weight" or "Abundance") and demonstrate basic filtering, normalization, and grouping operations. This prepares the data for further analysis.

In [ ]:
# Example: pick the primary tabular record set (first available RecordSet)
focus_recset = record_sets_ids[0]
df = dataframes[focus_recset]

print(f"Working with RecordSet: {focus_recset}, columns: {df.columns.tolist()}")

# Guess numeric columns by dtype or name (users should check/correct field names)
numeric_candidates = [c for c in df.columns if df[c].dtype.kind in 'fi']
if not numeric_candidates and len(df) > 0:
    # Try to convert string columns that might contain numbers
    for c in df.columns:
        try:
            sampleval = df[c].iloc[0]
            float(sampleval)
            numeric_candidates.append(c)
        except:
            pass
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field for filtering: {numeric_field}")
else:
    print('No obvious numeric field found. Please edit and select an appropriate field.')
    numeric_field = None

if numeric_field is not None:
    # Drop NA for the field to ensure filtering works
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > df[numeric_field].quantile(0.75)]  # Example: keep top 25%
    print(f"Records with {numeric_field} above 75th percentile:")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    normed_col = f"{numeric_field}_normalized"
    filtered_df[normed_col] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normed_col]].head())

    # Try grouping by a non-numeric field (e.g. 'description', or pick next available column)
    group_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == 'O']
    group_field = group_candidates[0] if group_candidates else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print('Skipping EDA, since no numeric field was detected.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field (e.g. a boxplot and histogram). We'll use matplotlib for simplicity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and not df[numeric_field].isnull().all():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field].dropna(), ax=axes[0], kde=True)
    axes[0].set_title(f'Histogram of {numeric_field}')
    sns.boxplot(x=df[numeric_field].dropna(), ax=axes[1])
    axes[1].set_title(f'Boxplot of {numeric_field}')
    plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion

- The FAIR^2 dataset schema provides tabular protein data at the record set granularity, suitable for stats and feature analysis.
- We demonstrated how to use `mlcroissant` to programmatically enumerate record sets, load tabular files, and extract data by field `@id`.
- After loading, we performed simple EDA and normalization; users can extend this with domain-specific protein analysis or ML workflows as needed.
